# Data center workbook: what is in it?

Reads `Datacenter Table Structure (v3).xlsx` tab by tab and reports structure, value quality, ID types, duplicates across tabs, and graph shape.

- pandas only: no Spark, no database, nothing is written to disk.
- Upload the workbook next to this notebook (or set `XLSX_PATH` in the first cell), then Run All. The load cell can take a minute or two.
- Every cell gives one HTML output with small tables, sized to fit a screenshot.
- Names below are the Excel headers; the short names in brackets are used only inside this notebook.

In [ ]:
# ── 0. Setup ────────────────────────────────────────────────────────────────
import re, html, warnings, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import HTML, display

warnings.filterwarnings("ignore")

XLSX_PATH = "Datacenter Table Structure (v3).xlsx"   # workbook next to this notebook, or a full path
TOP_N     = 15                                       # rows in "top / example" tables (small, so outputs fit a screenshot)

try:
    import openpyxl
    from openpyxl.utils import get_column_letter, column_index_from_string
except ImportError:
    raise ImportError("openpyxl is missing: run `pip install openpyxl` in a terminal, restart the kernel, rerun.")
if not Path(XLSX_PATH).exists():
    raise FileNotFoundError(f"Workbook not found at '{XLSX_PATH}'. Upload it next to this notebook or fix XLSX_PATH.")

# Values allowed by the workbook's own "Drop Down Options" tab (copied from that tab).
ALLOWED = {
    "pnc_flag":  ["Y", "N"],
    "so_type":   ["Supplier", "Customer", "Financing", "Equity", "Data Center"],
    "so_assign": ["Delineated", "Non-Delineated"],
    "so_offbs":  ["Leases not yet commenced", "Purchase commitments", "Guarantees", "Not off B/S"],
}
DEFAULT_DC_IDS = {"DCM9999999", "DCC9999999"}        # the "Default DC" bucket used by Scorecards / Survey

# ── display helper: ONE consolidated HTML output per cell ───────────────────
_CSS = """<style>
.dcx h4{margin:14px 0 4px 0;font-family:sans-serif;font-size:13px}
.dcx table{border-collapse:collapse;font-size:12px;font-family:sans-serif}
.dcx th{background:#1f4e79;color:#fff;padding:3px 8px;text-align:left}
.dcx td{padding:2px 8px;border-bottom:1px solid #ddd;white-space:nowrap}
.dcx p{font-family:sans-serif;font-size:12px;margin:2px 0}
</style>"""

def _fmt(df, width=72):
    out = df.copy()
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if pd.api.types.is_numeric_dtype(s):
            if str(c).endswith("%"):
                out[c] = s.map(lambda x: "" if pd.isna(x) else f"{x:.1%}")
            else:
                out[c] = s.map(lambda x: "" if pd.isna(x) else
                               (f"{x:,.0f}" if (abs(x) >= 100 or float(x).is_integer()) else f"{x:,.3f}"))
        else:
            out[c] = s.map(lambda x: "" if pd.isna(x) else (str(x) if len(str(x)) <= width else str(x)[:width - 1] + "…"))
    return out

def show(*blocks):
    """blocks = (title, DataFrame or text). Renders everything as a single HTML output."""
    parts = [_CSS, '<div class="dcx">']
    for title, obj in blocks:
        parts.append(f"<h4>{html.escape(str(title))}</h4>")
        if isinstance(obj, pd.DataFrame):
            parts.append("<p>(no rows)</p>" if obj.empty else _fmt(obj).to_html(index=False, border=0, na_rep=""))
        else:
            parts.append(f"<p>{html.escape(str(obj))}</p>")
    parts.append("</div>")
    display(HTML("".join(parts)))

# ── small parsers ───────────────────────────────────────────────────────────
def _txt(v):
    """Cell value → trimmed text (None for empty). Whole-number floats lose the '.0'; dates become YYYY-MM-DD."""
    if v is None:
        return None
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        if np.isnan(v):
            return None
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (datetime.datetime, datetime.date)):
        return v.isoformat()[:10]
    s = str(v).strip()
    return s if s else None

def norm(v):
    """Header text → lower case, single spaces (headers in the workbook contain line breaks)."""
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else re.sub(r"\s+", " ", str(v)).strip().lower()

def clean_id(v):
    """Upper-case text id. Digit-only ids shorter than 10 are zero-padded (Excel drops leading zeros on numbers)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = str(v).strip().upper()
    if not s:
        return np.nan
    return s.zfill(10) if (s.isdigit() and len(s) < 10) else s

def id_kind(s):
    if not isinstance(s, str):
        return "blank"
    if s in DEFAULT_DC_IDS:
        return "Default DC bucket"
    if re.fullmatch(r"\d{10}", s):
        return "PNC Power ID (10 digits)"
    if s.startswith("DCM"):
        return "DCM (non-PNC DC master)"
    if s.startswith("DCC"):
        return "DCC (non-PNC DC entity)"
    if s.startswith("XX"):
        return "XX (placeholder id)"
    return "other"

def parse_pct(v):
    """→ (class, value as fraction). 'UKN' is the workbook's code for unknown."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ("blank", np.nan)
    s = str(v).strip()
    if s.upper() in ("UKN", "UNK", "UNKNOWN", "N/A", "NA"):
        return ("UKN", np.nan)
    try:
        x = float(s[:-1]) / 100 if s.endswith("%") else float(s.replace(",", ""))
    except ValueError:
        return ("other text", np.nan)
    if x > 1:
        return ("above 100% (check)", x)
    if x < 0:
        return ("negative (check)", x)
    return ("number", x)

def parse_amt(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = re.sub(r"[$,\s]", "", str(v))
    if s in ("", "-"):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

_LEGAL = {"INC", "INCORPORATED", "LLC", "LP", "LLP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PLC", "NA", "SA", "GMBH"}
def name_key(v):
    """Same idea as the workbook's 'Near-Duplicates' column: upper case, punctuation and legal words removed."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = re.sub(r"\(.*?\)", " ", str(v).upper()).replace(".", "").replace("&", " AND ")
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    k = "".join(t for t in s.split() if t not in _LEGAL)
    return k if k else np.nan

def find_header(df, keys, max_rows=15):
    """First row (within max_rows) where every key appears inside some cell."""
    for i in range(min(max_rows, len(df))):
        cells = [norm(v) for v in df.iloc[i].tolist()]
        if all(any(k in c for c in cells) for k in keys):
            return i
    return None

def as_table(sheet, hdr):
    """Sheet → table using row `hdr` as header. Returns (table with excel_row, {header: Excel column letter})."""
    df = RAW[sheet]
    heads, seen = [], {}
    for j, v in enumerate(df.iloc[hdr].tolist()):
        h = re.sub(r"\s+", " ", str(v)).strip() if isinstance(v, str) else f"(no header, col {get_column_letter(j + 1)})"
        seen[h] = seen.get(h, 0) + 1
        heads.append(h if seen[h] == 1 else f"{h} [{seen[h]}]")
    body = df.iloc[hdr + 1:].copy()
    body.columns = heads
    body = body.dropna(how="all")
    body = body[[c for c in body.columns if not (c.startswith("(no header") and body[c].isna().all())]]
    letters = {h: get_column_letter(j + 1) for j, h in enumerate(heads)}
    body.insert(0, "excel_row", (body.index + 1).astype(str))
    return body.reset_index(drop=True), letters

# ── load the workbook once (all cells as text so Power IDs keep their leading zeros) ──
def extent(ws):
    """Last row / column that holds a value (formatting-only cells below or right of the data are ignored)."""
    cells = getattr(ws, "_cells", None)
    if not cells:
        return ws.max_row, ws.max_column
    used = [(r, c) for (r, c), cell in cells.items() if cell.value is not None]
    return (max(r for r, _ in used), max(c for _, c in used)) if used else (0, 0)

WB  = openpyxl.load_workbook(XLSX_PATH, data_only=True)
RAW, EXTENT = {}, {}
for ws in WB.worksheets:
    nr, nc = EXTENT[ws.title] = extent(ws)
    rows = [[_txt(v) for v in r] for r in ws.iter_rows(min_row=1, max_row=nr, max_col=nc, values_only=True)] if nr else []
    RAW[ws.title] = pd.DataFrame(rows) if rows else pd.DataFrame()

show(("Loaded", f"{len(RAW)} sheets from {Path(XLSX_PATH).name}: " + ", ".join(RAW)))

## 1. Workbook inventory
Every sheet, including hidden ones: size, hidden columns, hidden rows (rows hidden by a filter count here) and whether it has the relationship layout (master → data center entity → second order → third order).

In [ ]:
# ── 1. Inventory ────────────────────────────────────────────────────────────
def hidden_cols(ws):
    out = set()
    for d in ws.column_dimensions.values():
        if d.hidden:
            lo = d.min or column_index_from_string(d.index)
            hi = min(d.max or lo, EXTENT[ws.title][1])
            out.update(range(lo, hi + 1))
    return [get_column_letter(k) for k in sorted(out)]

REL_HDR = {}                                   # relationship tabs → header row index
inv = []
for ws in WB.worksheets:
    df  = RAW[ws.title]
    hdr = find_header(df, ["master power id", "second order"])
    if hdr is not None:
        REL_HDR[ws.title] = hdr
    hc = hidden_cols(ws)
    inv.append({
        "sheet": ws.title,
        "state": ws.sheet_state,
        "rows used": EXTENT[ws.title][0],
        "cols used": EXTENT[ws.title][1],
        "hidden cols": ", ".join(hc) if hc else "",
        "hidden rows": sum(1 for d in ws.row_dimensions.values() if d.hidden),
        "filter": ws.auto_filter.ref or "",
        "relationship layout": "yes (header on row %d)" % (hdr + 1) if hdr is not None else "",
    })
show(("1. Sheets in the workbook", pd.DataFrame(inv)))

## 2. Header map
The relationship tabs do not share one column order (some have extra or hidden columns), so columns are matched by header text, not by position.
First table: Excel column letter where each field was found per tab ("—" = not found). Second table: columns that are in a tab but not part of the common layout, with what they contain.

In [ ]:
# ── 2. Header map → common layout ───────────────────────────────────────────
def _rules():
    r = [("pnc_flag",    lambda h: "pnc flag" in h),
         ("master_id",   lambda h: "master power" in h),
         ("master_name", lambda h: "master name" in h),
         ("dc_id",       lambda h: h.startswith("datacenter customer power")),
         ("dc_name",     lambda h: h.startswith("datacenter customer name"))]
    for p, lab in (("so", "second order"), ("to", "third order")):
        r += [(f"{p}_id",        lambda h, lab=lab: h.startswith(lab + " power")),
              (f"{p}_name",      lambda h, lab=lab: h.startswith(lab + " customer name")),
              (f"{p}_industry",  lambda h, lab=lab: h.startswith(lab + " customer industry")),
              (f"{p}_type",      lambda h, lab=lab: h.startswith(lab + " relationship type")),
              (f"{p}_assign",    lambda h, lab=lab: h.startswith(lab + " relationship assignment")),
              (f"{p}_pct",       lambda h, lab=lab: h.startswith(lab + " relationship")),      # after type / assignment
              (f"{p}_loan_type", lambda h, lab=lab: h.startswith(lab + " loan type")),
              (f"{p}_amount",    lambda h, lab=lab: h.startswith(lab + " loan amount")),
              (f"{p}_offbs",     lambda h, lab=lab: h.startswith(lab + " off"))]
    r += [("data_source", lambda h: h.startswith("data source")),
          ("as_of_date",  lambda h: h.startswith("as of")),
          ("analyst",     lambda h: h.startswith("analyst")),
          ("notes",       lambda h: h.startswith("notes"))]
    return r
RULES  = _rules()
FIELDS = [f for f, _ in RULES]
LABEL  = {"pnc_flag": "Datacenter PNC Flag", "master_id": "Datacenter Master Power ID", "master_name": "Datacenter Master Name",
          "dc_id": "Datacenter Customer Power ID", "dc_name": "Datacenter Customer Name",
          "data_source": "Data Source", "as_of_date": "As of Date", "analyst": "Analyst", "notes": "Notes"}
for p, lab in (("so", "Second Order"), ("to", "Third Order")):
    LABEL.update({f"{p}_id": f"{lab} Power ID", f"{p}_name": f"{lab} Customer Name", f"{p}_industry": f"{lab} Customer Industry",
                  f"{p}_type": f"{lab} Relationship Type", f"{p}_assign": f"{lab} Relationship Assignment",
                  f"{p}_pct": f"{lab} Relationship %", f"{p}_loan_type": f"{lab} Loan Type",
                  f"{p}_amount": f"{lab} Loan Amount", f"{p}_offbs": f"{lab} Off-Balance Sheet"})

frames, where, extras = [], {}, []
for tab, hdr in REL_HDR.items():
    t, letters = as_table(tab, hdr)
    hid = set(hidden_cols(WB[tab]))
    mapping = {}                                        # field → header
    for h in t.columns[1:]:
        f = next((f for f, test in RULES if test(norm(h))), None)
        if f is not None and f not in mapping:
            mapping[f] = h
        else:
            vals = t[h].dropna()
            extras.append({"tab": tab, "col": letters[h], "hidden": "yes" if letters[h] in hid else "",
                           "header": h, "filled rows": len(vals), "distinct": vals.nunique(),
                           "examples": " | ".join(vals.drop_duplicates().head(3))})
    where[tab] = {f: letters[mapping[f]] for f in mapping}
    out = pd.DataFrame({f: (t[mapping[f]] if f in mapping else np.nan) for f in FIELDS})
    out.insert(0, "excel_row", t["excel_row"])
    out.insert(0, "tab", tab)
    frames.append(out)

if not frames:
    raise RuntimeError("No sheet with the relationship layout was found (looked for 'Master Power ID' + 'Second Order' headers). "
                       "Check section 1 and the sheet names.")
E = pd.concat(frames, ignore_index=True)
TABS = list(REL_HDR)

# typed helper columns (raw text columns stay untouched)
for c in ("master_id", "dc_id", "so_id", "to_id"):
    E[c + "_raw"] = E[c]
    E[c] = E[c].map(clean_id)
E["amount"] = E["so_amount"].map(parse_amt)
_p = E["so_pct"].map(parse_pct)
E["pct_class"], E["pct"] = _p.str[0], _p.str[1]
E["pct_shown"] = np.where(E.pct_class == "number", E.pct.map(lambda x: f"{x:.1%}" if pd.notna(x) else ""), E.so_pct)   # numbers as %, anything else as stored
for f in FIELDS:
    E[f] = E[f].astype(object)
E["so_kind"] = E["so_id"].map(id_kind)

hm = pd.DataFrame({"field": [LABEL[f] for f in FIELDS]})
for tab in TABS:
    hm[tab] = [where[tab].get(f, "—") for f in FIELDS]
show(("2a. Where each field sits (Excel column letter per tab)", hm),
     ("2b. Columns outside the common layout", pd.DataFrame(extras)))

## 3. Size of each relationship tab
One row in a tab = one link between a data center entity and a second-order party (supplier, customer, lender ...), optionally extended to a third-order party.

In [ ]:
# ── 3. Tab profile ──────────────────────────────────────────────────────────
prof = []
for tab in TABS:
    d = E[E.tab == tab]
    prof.append({
        "tab": tab, "rows": len(d),
        "PNC flag Y": int((d.pnc_flag.str.upper() == "Y").sum()), "PNC flag N": int((d.pnc_flag.str.upper() == "N").sum()),
        "masters": d.master_id.nunique(), "DC entities": d.dc_id.nunique(), "second-order parties": d.so_id.nunique(),
        "rows with amount": int(d.amount.notna().sum()), "rows with numeric pct": int((d.pct_class == "number").sum()),
        "rows with third order": int((d.to_id.notna() | d.to_name.notna()).sum()),
        "rows missing 2nd-order id": int(d.so_id.isna().sum()),
    })
prof = pd.DataFrame(prof)
tot = {"tab": "ALL TABS", "rows": len(E), "PNC flag Y": prof["PNC flag Y"].sum(), "PNC flag N": prof["PNC flag N"].sum(),
       "masters": E.master_id.nunique(), "DC entities": E.dc_id.nunique(), "second-order parties": E.so_id.nunique(),
       "rows with amount": prof["rows with amount"].sum(), "rows with numeric pct": prof["rows with numeric pct"].sum(),
       "rows with third order": prof["rows with third order"].sum(), "rows missing 2nd-order id": prof["rows missing 2nd-order id"].sum()}
show(("3. Relationship tabs (distinct counts in the ALL TABS row are de-duplicated across tabs)",
      pd.concat([prof, pd.DataFrame([tot])], ignore_index=True)))

## 4. Coded values against the drop-down lists
Counts per tab of each value in the coded columns. Status "case variant" = same word with different capitalisation; "NOT in list" = not offered by the Drop Down Options tab.

In [ ]:
# ── 4. Coded values ─────────────────────────────────────────────────────────
rows = []
for f, allowed in ALLOWED.items():
    vc = E.assign(_v=E[f].fillna("(blank)")).groupby(["_v", "tab"]).size().unstack(fill_value=0).reindex(columns=TABS, fill_value=0)
    low = [a.lower() for a in allowed]
    for val, r in vc.iterrows():
        status = ("ok" if val in allowed else "blank" if val == "(blank)"
                  else "case variant" if val.lower() in low else "NOT in list")
        rows.append({"field": LABEL[f], "value": val, "status": status, **{t: int(r[t]) for t in TABS}})
vals = pd.DataFrame(rows)
show(("4. Coded values per tab", vals))

## 5. Relationship % and amount: what do they hold in each tab?
The same two columns are used by every source, so this shows their range per tab and relationship type. Very different ranges across tabs mean the column does not carry one single meaning.

In [ ]:
# ── 5. Relationship % and amount by tab × relationship type ─────────────────
E["_type"] = E.so_type.fillna("(blank)").str.strip().str.title()
g = E.groupby(["tab", "_type"], sort=False)
pct_tbl = g.agg(rows=("pct_class", "size"),
                UKN=("pct_class", lambda s: int((s == "UKN").sum())),
                blank=("pct_class", lambda s: int((s == "blank").sum())),
                numeric=("pct_class", lambda s: int((s == "number").sum())),
                check=("pct_class", lambda s: int(s.str.contains("check|text").sum())),
                ).reset_index()
valid = E[E.pct_class == "number"].groupby(["tab", "_type"]).pct.agg(["min", "median", "max"]).reset_index()
valid.columns = ["tab", "_type", "min %", "median %", "max %"]
pct_tbl = pct_tbl.merge(valid, on=["tab", "_type"], how="left").rename(columns={"_type": "relationship type", "check": "to check"})

amt_tbl = g.agg(rows=("amount", "size"), with_amount=("amount", "count"),
                min=("amount", "min"), median=("amount", "median"), max=("amount", "max"), total=("amount", "sum")).reset_index()
amt_tbl = amt_tbl.rename(columns={"_type": "relationship type", "with_amount": "with amount"})
lt = E.assign(_l=E.so_loan_type.fillna("(blank)").str.strip()).groupby(["tab", "_l"]).size().unstack(fill_value=0).reindex(TABS).rename_axis(None, axis=1).reset_index()

odd = E[E.pct_class.isin(["above 100% (check)", "negative (check)", "other text"])]
odd_tbl = (odd[["tab", "excel_row", "dc_name", "so_name", "so_type", "so_pct", "pct_class"]].head(TOP_N)
             .rename(columns={"so_pct": "stored value (1 = 100%)"}))
show(("5a. Relationship %", pct_tbl),
     ("5b. Amount (Second Order Loan Amount)", amt_tbl),
     ("5c. Second Order Loan Type per tab (row counts)", lt),
     (f"5d. Relationship % entries to check: {len(odd):,} rows (first {TOP_N})", odd_tbl))

## 6. ID types: how much of this can link to payment data?
Only 10-digit Power IDs can be looked up in PNC data. DCM / DCC ids are the workbook's own ids for non-PNC data centers, XX ids are placeholders, and DCM9999999 / DCC9999999 is the "Default DC" bucket.
Coverage is given both as number of parties and as share of the amount column.

In [ ]:
# ── 6. ID anatomy ───────────────────────────────────────────────────────────
roles = {"Datacenter master": "master_id", "Datacenter entity (borrower)": "dc_id", "Second-order party": "so_id", "Third-order party": "to_id"}
kinds = ["PNC Power ID (10 digits)", "DCM (non-PNC DC master)", "DCC (non-PNC DC entity)", "XX (placeholder id)", "Default DC bucket", "other"]
r1 = []
for lab, c in roles.items():
    ids = pd.Series(E[c].dropna().unique())
    k = ids.map(id_kind).value_counts()
    raw_ = E[c + "_raw"].dropna().astype(str).str.strip()
    r1.append({"role": lab, "distinct ids": len(ids), **{x: int(k.get(x, 0)) for x in kinds},
               "rows zero-padded here": int((raw_.str.isdigit() & (raw_.str.len() < 10)).sum())})

r2 = []
for tab in TABS + ["ALL TABS"]:
    d = E if tab == "ALL TABS" else E[E.tab == tab]
    n_all = d.so_id.nunique()
    n_pid = d.loc[d.so_kind == "PNC Power ID (10 digits)", "so_id"].nunique()
    a_all = d.amount.sum()
    a_pid = d.loc[d.so_kind == "PNC Power ID (10 digits)", "amount"].sum()
    r2.append({"tab": tab, "second-order parties": n_all, "with PNC Power ID": n_pid,
               "parties %": n_pid / n_all if n_all else np.nan,
               "amount total": a_all, "amount on PNC Power IDs": a_pid, "amount %": a_pid / a_all if a_all else np.nan})
show(("6a. Distinct ids by role and id type (all relationship tabs)", pd.DataFrame(r1)),
     ("6b. Second-order parties that carry a real PNC Power ID, by tab", pd.DataFrame(r2)))

## 7. ID Master List
The workbook's node list. Checks: one id per company, one company per id, and whether every id used in the relationship tabs is listed.

In [ ]:
# ── 7. ID Master List ───────────────────────────────────────────────────────
ML, blocks = None, []
for sheet, df in RAW.items():
    hdr = find_header(df, ["company name", "power id"])
    if hdr is None or sheet in REL_HDR:
        continue
    t, _ = as_table(sheet, hdr)
    col = {}
    for h in t.columns[1:]:
        n = norm(h)
        if n.startswith("company name"):  col.setdefault("name", h)
        elif n.startswith("near"):        col.setdefault("key", h)
        elif n == "power id":             col.setdefault("id", h)
        elif n.startswith("classification"): col.setdefault("class", h)
        elif n.startswith("ticker"):      col.setdefault("ticker", h)
    if "name" not in col or "id" not in col:
        continue
    m = pd.DataFrame({k: t[h] for k, h in col.items()})
    m["excel_row"] = t["excel_row"]
    m["id"] = m["id"].map(clean_id)
    if "key" not in m:   m["key"] = m["name"].map(name_key)
    if "class" not in m: m["class"] = np.nan
    m["kind"] = m["id"].map(id_kind)
    per_id  = m.dropna(subset=["id"]).groupby("id").name.nunique()
    per_key = m.dropna(subset=["id", "key"]).groupby("key").id.nunique()
    summ = pd.DataFrame([{"sheet": sheet, "rows": len(m), "distinct ids": m.id.nunique(), "blank ids": int(m.id.isna().sum()),
                          "ids with more than one company name": int((per_id > 1).sum()),
                          "name keys with more than one id": int((per_key > 1).sum()),
                          "with ticker": int(m["ticker"].notna().sum()) if "ticker" in m else 0}])
    xt = (m.assign(c=m["class"].fillna("(blank)")).groupby(["c", "kind"]).size().unstack(fill_value=0)
            .rename_axis(None, axis=1).reset_index().rename(columns={"c": "classification"}))
    blocks += [(f"7. '{sheet}': summary", summ), (f"'{sheet}': classification by id type (rows)", xt)]
    if ML is None:
        ML = m                                           # first matching sheet is used for the reconciliation below
        multi = per_key[per_key > 1].index
        ex = (m[m.key.isin(multi)].sort_values(["key", "id"]).groupby("key")
                .agg(names=("name", lambda s: " / ".join(s.drop_duplicates().head(3))),
                     ids=("id", lambda s: ", ".join(s.dropna().drop_duplicates().head(4)))).reset_index().head(TOP_N))
        blocks.append((f"'{sheet}': same name key under more than one id (first {TOP_N})", ex))

if ML is None:
    blocks.append(("7. ID Master List", "No sheet with 'Company name' + 'Power ID' headers was found."))
else:
    known = set(ML.id.dropna())
    rec = []
    for lab, c in roles.items():
        used = set(E[c].dropna())
        rec.append({"role": lab, "ids used in relationship tabs": len(used), "missing from master list": len(used - known)})
    used_all = set().union(*[set(E[c].dropna()) for c in roles.values()])
    blocks.append(("Ids used in the relationship tabs vs the master list", pd.DataFrame(rec)))
    blocks.append(("Master-list ids never used in a relationship tab", f"{len(known - used_all):,} of {len(known):,}"))
show(*blocks)

## 8. Duplicates and conflicts across tabs
The email says the same information may sit in more than one tab. A relationship is identified here by: data center entity id + second-order id + relationship type.

In [ ]:
# ── 8. Duplicates ───────────────────────────────────────────────────────────
K = E.dropna(subset=["dc_id", "so_id"]).copy()
K["key"] = K.dc_id + " | " + K.so_id + " | " + K["_type"]
gk = K.groupby("key").agg(
    rows=("tab", "size"), n_tabs=("tab", "nunique"),
    tabs=("tab", lambda s: " + ".join(sorted(set(s)))),
    n_amt=("amount", lambda s: s.dropna().round(0).nunique()),
    n_pct=("pct", lambda s: s.dropna().round(4).nunique()),
    dc=("dc_name", "first"), party=("so_name", "first"), type=("_type", "first"),
    amounts=("amount", lambda s: " / ".join(f"{x:,.0f}" for x in s.dropna().round(0).drop_duplicates().head(3))),
    pcts=("pct_shown", lambda s: " / ".join(s.dropna().drop_duplicates().head(3))))
cross = gk[gk.n_tabs > 1]
by_combo = (cross.groupby("tabs").agg(relationships=("rows", "size"),
                                      amount_conflicts=("n_amt", lambda s: int((s > 1).sum())),
                                      pct_conflicts=("n_pct", lambda s: int((s > 1).sum()))).reset_index()
            .rename(columns={"tabs": "tab combination", "amount_conflicts": "different amounts", "pct_conflicts": "different pct"}))
within = (K.groupby(["tab", "key"]).size().reset_index(name="n").query("n > 1").groupby("tab")
            .agg(repeated_relationships=("n", "size"), extra_rows=("n", lambda s: int((s - 1).sum()))).reindex(TABS, fill_value=0).reset_index()
            .rename(columns={"repeated_relationships": "relationships repeated", "extra_rows": "extra rows"}))
ex_cross = (cross.sort_values(["n_amt", "n_pct"], ascending=False)[["dc", "party", "type", "tabs", "amounts", "pcts"]].head(TOP_N))

S = E.dropna(subset=["so_id"]).assign(nk=lambda d: d.so_name.map(name_key))
ids_per_name = S.dropna(subset=["nk"]).groupby("nk").so_id.nunique()
names_per_id = S.dropna(subset=["nk"]).groupby("so_id").nk.nunique()
multi = ids_per_name[ids_per_name > 1].index
ex_name = (S[S.nk.isin(multi)].groupby("nk")
             .agg(name=("so_name", "first"), ids=("so_id", lambda s: ", ".join(s.drop_duplicates().head(4))),
                  tabs=("tab", lambda s: " + ".join(sorted(set(s))))).reset_index(drop=True).head(TOP_N))
show(("8a. Same relationship found in more than one tab", by_combo),
     (f"8b. Examples (conflicting ones first, first {TOP_N})", ex_cross),
     ("8c. Same relationship repeated inside one tab", within),
     ("8d. Second-order names and ids",
      f"{len(multi):,} names appear under more than one id; {int((names_per_id > 1).sum()):,} ids appear under more than one name."),
     (f"8e. Same second-order name under more than one id (first {TOP_N})", ex_name))

## 9. Graph shape
What the workbook looks like as a network: how many nodes and links, which second-order parties are shared by many data centers (these are the channels through which stress would travel), and which data centers have the most links.

In [ ]:
# ── 9. Graph shape ──────────────────────────────────────────────────────────
own  = E.dropna(subset=["master_id", "dc_id"])[["master_id", "dc_id"]].drop_duplicates()
rel  = K.drop_duplicates("key")
thr  = E.dropna(subset=["so_id", "to_id"])[["so_id", "to_id"]].drop_duplicates()
nodes = set(E.master_id.dropna()) | set(E.dc_id.dropna()) | set(E.so_id.dropna()) | set(E.to_id.dropna())
shape = pd.DataFrame([
    {"item": "nodes (distinct ids, all roles)", "count": len(nodes)},
    {"item": "master → data center entity links", "count": len(own)},
    {"item": "   of which master and entity share one id", "count": int((own.master_id == own.dc_id).sum())},
    {"item": "data center entity → second-order links (distinct)", "count": len(rel)},
    {"item": "second-order → third-order links (distinct)", "count": len(thr)},
    {"item": "ids that are a data center in one row and a second-order party in another", "count":
        len((set(E.master_id.dropna()) | set(E.dc_id.dropna())) & set(E.so_id.dropna()))},
])
by_type = rel.groupby("_type").size().reset_index(name="distinct links").rename(columns={"_type": "relationship type"})

real = S[~S.master_id.isin(DEFAULT_DC_IDS)]
sp = real.groupby("so_id").agg(name=("so_name", "first"), masters=("master_id", "nunique"), dc_entities=("dc_id", "nunique"),
                               types=("_type", lambda s: ", ".join(sorted(set(s)))),
                               tabs=("tab", lambda s: " + ".join(sorted(set(s)))), amount=("amount", "sum"))
bins = pd.cut(sp.masters, [0, 1, 2, 5, 10**9], labels=["1 master", "2 masters", "3 to 5", "6 or more"])
dist = bins.value_counts().reindex(["1 master", "2 masters", "3 to 5", "6 or more"]).reset_index()
dist.columns = ["second-order party is linked to", "parties"]
top_sp = (sp.sort_values(["masters", "amount"], ascending=False).head(TOP_N).reset_index()
            .rename(columns={"so_id": "second-order id", "dc_entities": "DC entities", "amount": "amount (sum)"}))

mm = E.dropna(subset=["master_id"]).groupby("master_id").agg(
        master=("master_name", "first"), pnc=("pnc_flag", "first"), dc_entities=("dc_id", "nunique"), parties=("so_id", "nunique"),
        customers=("_type", lambda s: int((s == "Customer").sum())), suppliers=("_type", lambda s: int((s == "Supplier").sum())),
        financing=("_type", lambda s: int((s == "Financing").sum())), tabs=("tab", lambda s: " + ".join(sorted(set(s)))))
top_m = (mm.sort_values("parties", ascending=False).head(TOP_N).reset_index()
           .rename(columns={"master_id": "master id", "pnc": "PNC flag", "dc_entities": "DC entities", "parties": "second-order parties",
                            "customers": "customer rows", "suppliers": "supplier rows", "financing": "financing rows"}))
show(("9a. Size", shape), ("9b. Distinct links by relationship type", by_type),
     ("9c. How many masters each second-order party touches (Default DC bucket excluded)", dist),
     (f"9d. Most shared second-order parties (top {TOP_N})", top_sp),
     (f"9e. Masters with the most second-order parties (top {TOP_N})", top_m))

## 10. Renewables and Hyperscalers indirect
Renewables has its own layout: PNC lends to a project borrower, the project sells power to an off-taker. Per the email, "Secondary Exposure to Off-Taker" is treated as off-balance-sheet exposure to the off-taker when Project Type is Renewables.
The sheet has names only, so off-takers are matched to the ID Master List by name key.

In [ ]:
# ── 10. Renewables + Hyperscalers indirect ──────────────────────────────────
blocks = []
ren_sheet = next((s for s, df in RAW.items() if find_header(df, ["off-taker", "borrower"]) is not None and s not in REL_HDR), None)
R = None
if ren_sheet is None:
    blocks.append(("10. Renewables", "No sheet with 'Off-taker' + 'Borrower' headers was found."))
else:
    t, _ = as_table(ren_sheet, find_header(RAW[ren_sheet], ["off-taker", "borrower"]))
    pick = {}
    for h in t.columns[1:]:
        n = norm(h)
        if n == "off-taker":                      pick.setdefault("offtaker", h)
        elif n.startswith("capital type"):        pick.setdefault("capital", h)
        elif n.startswith("project type"):        pick.setdefault("project_type", h)
        elif n.startswith("contract type"):       pick.setdefault("contract", h)
        elif n.startswith("site"):                pick.setdefault("site", h)
        elif n == "borrower":                     pick.setdefault("borrower", h)
        elif n == "sponsor":                      pick.setdefault("sponsor", h)
        elif n.startswith("contracted power"):    pick.setdefault("mw", h)
        elif n.startswith("percentage of off"):   pick.setdefault("share", h)
        elif n.startswith("dhe"):                 pick.setdefault("dhe", h)
        elif n.startswith("secondary exposure"):  pick.setdefault("secondary", h)
        elif "expiration" in n:                   pick.setdefault("expiry", h)
    need = ["offtaker", "borrower", "share", "dhe", "secondary"]
    miss = [k for k in need if k not in pick]
    if miss:
        blocks.append(("10. Renewables", f"Sheet '{ren_sheet}' found but these columns were not: {miss}. Headers: {list(t.columns[1:])}"))
    else:
        R = pd.DataFrame({k: t[h] for k, h in pick.items()})
        for k in ("capital", "project_type", "contract", "site", "sponsor", "mw", "expiry"):
            if k not in R: R[k] = np.nan
        R["dhe_n"], R["sec_n"], R["mw_n"] = R.dhe.map(parse_amt), R.secondary.map(parse_amt), R.mw.map(parse_amt)
        R["share_n"] = R.share.map(lambda v: parse_pct(v)[1])
        R = R[R.offtaker.notna()].copy()
        calc = R.share_n * R.dhe_n
        ok = ((calc - R.sec_n).abs() <= 0.02 * R.sec_n.abs().clip(lower=1))
        dhe_b = R.dropna(subset=["borrower"]).groupby("borrower").dhe_n
        summ = pd.DataFrame([
            {"item": "rows (one off-take contract each)", "value": f"{len(R):,}"},
            {"item": "off-takers / borrowers / sponsors", "value": f"{R.offtaker.nunique():,} / {R.borrower.nunique():,} / {R.sponsor.nunique():,}"},
            {"item": "Secondary Exposure to Off-Taker, total", "value": f"{R.sec_n.sum():,.0f}"},
            {"item": "rows where secondary = share x DHE (within 2%)", "value": f"{int(ok.sum()):,} of {int(R.sec_n.notna().sum()):,} with a value"},
            {"item": "DHE at borrower level, counted once per borrower", "value": f"{dhe_b.max().sum():,.0f}"},
            {"item": "borrowers showing more than one DHE value", "value": f"{int((dhe_b.nunique() > 1).sum()):,}"},
            {"item": "contract expiry range", "value": f"{R.expiry.dropna().min()} to {R.expiry.dropna().max()}" if R.expiry.notna().any() else ""},
        ])
        pt = (R.assign(pt=R.project_type.fillna("(blank)"), cap=R.capital.fillna("(blank)")).groupby(["pt", "cap"])
                .agg(rows=("sec_n", "size"), secondary=("sec_n", "sum")).reset_index()
                .rename(columns={"pt": "Project Type", "cap": "Capital Type", "secondary": "secondary exposure"}))
        ot = (R.groupby("offtaker").agg(contracts=("sec_n", "size"), borrowers=("borrower", "nunique"), mw=("mw_n", "sum"),
                                        secondary=("sec_n", "sum")).sort_values("secondary", ascending=False).reset_index())
        if ML is not None:
            k2id = ML.dropna(subset=["key", "id"]).drop_duplicates("key").set_index("key").id
            ot["master-list id"] = ot.offtaker.map(name_key).map(k2id)
            n_hit = int(ot["master-list id"].notna().sum())
            match_txt = (f"{n_hit:,} of {len(ot):,} off-takers match a master-list company by name key, "
                         f"covering {ot.loc[ot['master-list id'].notna(), 'secondary'].sum() / ot.secondary.sum():.1%} of secondary exposure."
                         if ot.secondary.sum() else f"{n_hit:,} of {len(ot):,} off-takers match.")
        else:
            match_txt = "ID Master List not available, no matching done."
        ot = ot.rename(columns={"offtaker": "off-taker", "mw": "contracted MW", "secondary": "secondary exposure"})
        blocks += [(f"10a. '{ren_sheet}': summary", summ), ("10b. By Project Type and Capital Type", pt),
                   ("10c. Off-taker match to the ID Master List", match_txt),
                   (f"10d. Largest off-takers by secondary exposure (top {TOP_N})", ot.head(TOP_N))]

hyp = next((s for s in RAW if "hyperscaler" in s.lower()), None)
if hyp is None:
    blocks.append(("10e. Hyperscalers indirect", "Sheet not found."))
else:
    h = RAW[hyp].dropna(how="all").dropna(how="all", axis=1).head(40)
    h.columns = [get_column_letter(j + 1) for j in h.columns]
    h.insert(0, "row", h.index + 1)
    blocks.append((f"10e. '{hyp}' as it is (non-empty cells)", h.fillna("")))
show(*blocks)

## 11. Readiness numbers
The figures that decide how far the first draft can go: a graph view needs ids and links; a contagion calculation also needs a dependency weight (relationship %) and a comparable amount on most links.

In [ ]:
# ── 11. Readiness numbers ───────────────────────────────────────────────────
def share(a, b):
    return f"{a:,} of {b:,} ({a / b:.1%})" if b else "0 of 0"
n_rows = len(E)
so_ids = E.so_id.dropna().unique()
n_pid  = int((pd.Series(so_ids).map(id_kind) == "PNC Power ID (10 digits)").sum())
dc_ids = pd.Series(E.dc_id.dropna().unique())
dc_real = dc_ids[~dc_ids.isin(DEFAULT_DC_IDS)]
ready = [
    ("Relationship rows, all tabs", f"{n_rows:,}"),
    ("Distinct relationships after removing cross-tab repeats", f"{len(rel):,}"),
    ("Data center entities with a real PNC Power ID", share(int((dc_real.map(id_kind) == 'PNC Power ID (10 digits)').sum()), len(dc_real))),
    ("Second-order parties with a real PNC Power ID", share(n_pid, len(so_ids))),
    ("Rows with a numeric relationship %", share(int((E.pct_class == 'number').sum()), n_rows)),
    ("Rows with relationship % = UKN or blank", share(int(E.pct_class.isin(['UKN', 'blank']).sum()), n_rows)),
    ("Rows with an amount", share(int(E.amount.notna().sum()), n_rows)),
    ("Rows with BOTH numeric % and amount", share(int(((E.pct_class == 'number') & E.amount.notna()).sum()), n_rows)),
    ("Rows with a third-order party", share(int((E.to_id.notna() | E.to_name.notna()).sum()), n_rows)),
    ("Rows attached to the Default DC bucket (no specific data center)", share(int(E.dc_id.isin(DEFAULT_DC_IDS).sum()), n_rows)),
    ("Relationships present in more than one tab", f"{len(cross):,} (different amounts: {int((cross.n_amt > 1).sum()):,}, different %: {int((cross.n_pct > 1).sum()):,})"),
    ("Second-order names under more than one id", f"{len(multi):,}"),
    ("Coded values not matching the drop-down lists (rows)", f"{int(vals.loc[vals.status.isin(['case variant', 'NOT in list']), TABS].to_numpy().sum()):,}"),
]
if ML is not None:
    ready.append(("Ids used in relationship tabs but missing from the ID Master List", f"{len(used_all - known):,}"))
if R is not None:
    ready.append(("Renewables: off-take contracts / total secondary exposure", f"{len(R):,} / {R.sec_n.sum():,.0f}"))
show(("11. Readiness numbers", pd.DataFrame(ready, columns=["measure", "value"])))